In [1]:
import pandas as pd
import numpy as np
import os
import re
import iris
import cftime 
import gc
import glob
import datetime
import time 
from scipy.ndimage import label, generate_binary_structure

from functions import (get_rainfall_cube, load_3d_cube, prepare_flood_cube, filter_closer_to_catchment, subset_cube_to_bbox,
                      mask_cube_with_catchment_full_grid)
from functions_stage2 import (maybe_diagnose, get_data_at_peak_cell, plot_peak_check,get_rainfall_cube_subsection, 
                    plot_cluster_check)
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS # MOLLY_DIR_FF, RAINFALL_CSV_DIR, , ENSEMBLE_MEMBERS,  

# Get list of catchments to run
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())
files = os.listdir(OUT_DIR)
completed_catchments = {re.search(r'Catchment_(.+)', f).group(1) for f in files if re.search(r'Catchment_(.+)', f)}
catchments_to_run = (all_catchments - completed_catchments) - {105} - {14}

In [2]:
# Create a mask which masks out any cells not within the catchment
# This is year agnostic, and is applicable to all years
# Has shape 244, 180 (full GB)
# Create mask for whole country, and then trim it to the same extent as the rainfall cube before applying it
# This allows creating mask just once, and then applying for each year
# start_time1 = time.time()
# full_rain_cube = get_rainfall_cube(2015, '01', rainfall_cube_dir)   # any year, grid is same
# FULL_MASK_2D = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='full_cell')
# print(f"Creating a mask for EM{ens_num} in {round(time.time() - start_time1, 2)}s")   

In [3]:
catchment_num = "23"
def process_stage_2(catchment_num):
    catchment_name = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
    boundary_gdf   = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
    CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]

    # Get rainfall event data and apply pre-processing
    rainfall_events = pd.read_pickle(f"../../Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl")

    #      ## Do some value manipulation
    #     rainfall_events['values'] = rainfall_events['values'].apply(
    #         lambda x: np.fromstring(x.strip('[]'), sep=' '))
    if catchment_num == "23":
        rainfall_events['rainfall_peak_day'] = pd.to_datetime(
            rainfall_events['rainfall_peak_day'], format="%d/%m/%Y %H:%M")
    else:
        rainfall_events['rainfall_peak_day'] = pd.to_datetime(
            rainfall_events['rainfall_peak_day'], format="%Y-%m-%d %H:%M:%S")

    rainfall_events['rainfall_peak_360'] = rainfall_events['rainfall_peak_day'].apply(
        lambda d: cftime.Datetime360Day(d.year, d.month, d.day, d.hour, d.minute, d.second))
    rainfall_events['hydro_day_360'] = (
        (rainfall_events['rainfall_peak_day'].dt.month % 12) * 30
        + rainfall_events['rainfall_peak_day'].dt.day)
    rainfall_events['day_360'] = (
        (rainfall_events['rainfall_peak_day'].dt.month - 1) * 30
        + rainfall_events['rainfall_peak_day'].dt.day)
    rainfall_events['t_local']    = rainfall_events['t_global'] - rainfall_events['start_idx']
    rainfall_events['event_num']  = range(1, len(rainfall_events) + 1)
    rainfall_events['event_num_this_ens'] = rainfall_events.groupby('ens').cumcount() + 1

    start_time1 = time.time()
    full_rain_cube = get_rainfall_cube(2015, '01', f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/")   # any year, grid is same
    FULL_MASK_2D = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='full_cell')
    print(f"Creating a mask in {round(time.time() - start_time1, 2)}s")   

    # Get hydraulic conductivity data
    HC_CUBE      = iris.load(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/HydraulicConductivity/5km_{catchment_num}.nc")[0]
    HC_CUBE.data = np.where(FULL_MASK_2D, HC_CUBE.data, np.nan)
    HC_CUBE = filter_closer_to_catchment(HC_CUBE, CATCHMENT_POLY, plot=False)
    HC_DATA      = HC_CUBE.data  # realise once — shape (y, x)
    flood_dir    = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{catchment_num}/"

    n_days_ls      = [2, 3, 4, 5]
    all_results    = []
    DIAGNOSE_EVENT = []

    flood_numpy = {}  # will hold realised numpy arrays, keyed [depth]['area'/'vol']
    flood_cubes = {}  # still needed for maybe_diagnose (expects iris cubes)
    current_ens = None


    for (ens_num, year), group in rainfall_events[1:5].groupby(['ens', 'year']):

        rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
        print(f"Processing ens={ens_num}, year={year} ({len(group)} events)")

        # ── Reload flood cubes only when ensemble member changes ──────────────────
        if ens_num != current_ens:
            if flood_cubes:
                del flood_cubes, flood_numpy
                gc.collect()
            ens_flood_dir = f"{flood_dir}/Ens{ens_num}_{catchment_num}/"
            flood_cubes = {}
            flood_numpy = {}
            for depth in [10, 30]:
                area_cube = prepare_flood_cube(load_3d_cube(
                    f"{ens_flood_dir}/{depth}cm/flooded_area_5km_total_Ens{ens_num}_{catchment_num}_{depth}cm.nc"))
                area_cube.data = np.where(FULL_MASK_2D, area_cube.data, np.nan)
                area_cube = filter_closer_to_catchment(area_cube, CATCHMENT_POLY, plot=False)
                vol_cube  = prepare_flood_cube(load_3d_cube(
                    f"{ens_flood_dir}/{depth}cm/flooded_volume_5km_total_Ens{ens_num}_{catchment_num}_{depth}cm.nc"))
                vol_cube.data = np.where(FULL_MASK_2D, vol_cube.data, np.nan)
                vol_cube = filter_closer_to_catchment(vol_cube, CATCHMENT_POLY, plot=False)
                flood_cubes[depth] = {'area': area_cube, 'vol': vol_cube}  # iris, for diagnostics
                flood_numpy[depth] = {'area': area_cube.data, 'vol': vol_cube.data}  # numpy, for stats
            current_ens = ens_num

        # ── Load SM cube and realise to numpy once per (ens, year) ───────────────
        sm_dir  = f'/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/Ens_{ens_num}/'
        sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{year-1}1201-{year}1130_mrso.nc")[0]
        sm      = iris.load(sm_file)[0]
        sm.data = np.where(FULL_MASK_2D, sm.data, np.nan)
        sm = filter_closer_to_catchment(sm, CATCHMENT_POLY, plot=False)
        sm_data = sm.data  # realise once — shape (time, y, x)

        # Build time array once per group (only needs iris, done outside row loop)
        sm_example  = get_data_at_peak_cell(sm, group.iloc[0], 'x_idx', 'y_idx')
        sm_time_arr = np.array([cell.point for cell in sm_example.coord('time').cells()])

        start_time = time.time()

        for row in group.itertuples():

            # ── Diagnostics (still uses iris cubes for plotting) ──────────────────
            maybe_diagnose(
                row,
                event_num=row.event_num,
                condition=(row.event_num in DIAGNOSE_EVENT),
                sm=sm,
                HC_CUBE=HC_CUBE,
                flood_cubes=flood_cubes,
                catchment_poly=CATCHMENT_POLY,
                ens_num=ens_num,
                rainfall_cube_dir=rainfall_cube_dir)

            xi = int(row.x_idx)
            yi = int(row.y_idx)

            # ── SM stats — direct numpy indexing, no iris overhead ────────────────
            sm_at_peak_data   = sm_data[:, yi, xi]
            rainfall_peak_360 = row.rainfall_peak_360

            mean_sm_stats = {}
            for n_days in n_days_ls:
                delta       = datetime.timedelta(days=n_days)
                mask        = (sm_time_arr >= (rainfall_peak_360 - delta)) & (sm_time_arr < rainfall_peak_360)
                subset_data = sm_at_peak_data[mask]

                if subset_data.size == 0:
                    print(f"  Warning: empty SM window for event={row.event_num}, "
                          f"n_days={n_days}, peak={rainfall_peak_360}")
                    mean_sm_stats[f'mean_sm_{n_days}_before_event'] = float('nan')
                    continue

                mean_sm_stats[f'mean_sm_{n_days}_before_event'] = float(subset_data.mean())

            # ── Flood stats — direct numpy indexing ───────────────────────────────
            flood_stats = {}
            for depth in [10, 30]:
                flood_stats[f"{depth}cm_area"]   = flood_numpy[depth]['area'][row.event_num_this_ens - 1, yi, xi]
                flood_stats[f"{depth}cm_volume"] = flood_numpy[depth]['vol'][row.event_num_this_ens - 1, yi, xi]

            # ── HC stats — direct numpy indexing ─────────────────────────────────
            hc_stats = {'hc_at_peak': HC_DATA[yi, xi]}

            # ------------
            results = analyse_peak_event(
                                row,
                                neighbourhood_size=1,
                                threshold_levels=[0.5, 0.6, 0.8],
                                plot=False
                            )

            all_results.append(results)

        print(f"  Took: {round(time.time() - start_time, 2)}s")

        del sm, sm_data
        gc.collect()


    results_df = pd.DataFrame(all_results)
    return results_df

results_df = process_stage_2(catchment_num)

Creating a mask in 29.2s
Processing ens=01, year=1994 (2 events)


NameError: name 'analyse_peak_event' is not defined